# Gold Price Prediction using LSTM-Attention

An undergraduate Machine Learning project for predicting gold (GLD ETF) prices using LSTM networks with Multi-Head Self-Attention.

## Overview

This notebook implements a complete ML pipeline:
1. **Data Collection** - Fetch GLD ETF data from Yahoo Finance
2. **Feature Engineering** - Create 31 technical indicators (SMA, EMA, RSI, MACD, Bollinger Bands, etc.)
3. **Data Preprocessing** - Scaling and sequence creation for LSTM
4. **Model Training** - LSTM-Attention model with walk-forward cross-validation
5. **Evaluation** - Compare with baseline models (Linear Regression, Random Forest, SVM)
6. **Visualization** - Plot predictions, errors, and model comparison

## Model Architecture

```
Input (30 days x 31 features)
  -> LSTM(50 hidden)
    -> Dropout(0.2)
      -> Multi-Head Self-Attention (4 heads, key_dim=50)
        -> Residual + LayerNorm
          -> Dense(H=30 outputs)  <- direct multi-step
```

**Key Design Decisions:**
- **Target**: Daily returns (stationary) instead of raw prices
- **Validation**: Walk-forward CV (5 folds) - simulates real deployment
- **Lookback**: 30 trading days (~6 weeks of market context)
- **Forecast Horizon**: 30 days (direct multi-step prediction)

## Installation

Run this cell first to install required packages:

In [ ]:
!pip install yfinance pandas numpy scikit-learn torch matplotlib seaborn ta -q

## Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import yfinance as yf
from typing import Dict, List, Tuple, Optional

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

---
## Part 1: Data Collection Module

In [ ]:
def fetch_gold_data(ticker="GLD", start="2015-01-01", end=None, interval="1d"):
    print(f"Fetching {ticker} data from {start}...")
    gold = yf.Ticker(ticker)
    df = gold.history(start=start, end=end, interval=interval)
    df.columns = [col.lower() if isinstance(col, str) else '_'.join(col).lower() for col in df.columns]
    print(f"Retrieved {len(df)} records.")
    return df

def preprocess_data(df):
    df = df.copy().ffill().bfill()
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    df = df.sort_index()[~df.index.duplicated(keep='first')]
    print(f"Data shape after preprocessing: {df.shape}")
    return df

def split_data(df, train_ratio=0.8):
    split_idx = int(len(df) * train_ratio)
    return df.iloc[:split_idx], df.iloc[split_idx:]

---
## Part 2: Feature Engineering Module

In [ ]:
def add_technical_indicators(df):    df = df.copy()    # Moving Averages    df["sma_10"] = df["close"].rolling(window=10).mean()    df["sma_20"] = df["close"].rolling(window=20).mean()    df["sma_50"] = df["close"].rolling(window=50).mean()    df["ema_10"] = df["close"].ewm(span=10, adjust=False).mean()    df["ema_20"] = df["close"].ewm(span=20, adjust=False).mean()    df["sma_cross_10_20"] = df["sma_10"] - df["sma_20"]    df["sma_cross_20_50"] = df["sma_20"] - df["sma_50"]        # Returns and Volatility    df["returns"] = df["close"].pct_change()    df["log_returns"] = np.log(df["close"] / df["close"].shift(1))    df["volatility_10"] = df["returns"].rolling(window=10).std()    df["volatility_20"] = df["returns"].rolling(window=20).std()        # RSI    delta = df["close"].diff()    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()    rs = gain / loss    df["rsi_14"] = 100 - (100 / (1 + rs))        # MACD    df["macd"] = df["close"].ewm(span=12, adjust=False).mean() - df["close"].ewm(span=26, adjust=False).mean()    df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()    df["macd_hist"] = df["macd"] - df["macd_signal"]        # Bollinger Bands    df["bb_middle"] = df["close"].rolling(window=20).mean()    bb_std = df["close"].rolling(window=20).std()    df["bb_upper"] = df["bb_middle"] + (bb_std * 2)    df["bb_lower"] = df["bb_middle"] - (bb_std * 2)    df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["bb_middle"]    df["bb_position"] = (df["close"] - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"])        # Range features    df["hl_range"] = (df["high"] - df["low"]) / df["close"]    df["oc_range"] = (df["close"] - df["open"]) / df["open"]        # Volume    if "volume" in df.columns:        df["volume_sma_10"] = df["volume"].rolling(window=10).mean()        df["volume_ratio"] = df["volume"] / df["volume_sma_10"]        # Lag features    for lag in [1, 2, 3, 5, 10]:        df[f"close_lag_{lag}"] = df["close"].shift(lag)        df[f"returns_lag_{lag}"] = df["returns"].shift(lag)        # Rolling stats    for window in [5, 10, 20]:        df[f"close_mean_{window}"] = df["close"].rolling(window=window).mean()        df[f"close_std_{window}"] = df["close"].rolling(window=window).std()        # Time features    df["day_of_week"] = df.index.dayofweek    df["month"] = df.index.month    df["quarter"] = df.index.quarter        df = df.dropna()    print(f"Features added. Shape: {df.shape}")    return dfdef prepare_features(df, feature_columns=None):    if feature_columns is None:        feature_columns = ["open", "high", "low", "close", "volume", "sma_10", "sma_20", "ema_10", "ema_20",                          "returns", "volatility_10", "volatility_20", "rsi_14", "macd", "macd_signal", "macd_hist",                          "bb_width", "bb_position", "hl_range", "oc_range", "volume_ratio",                          "close_lag_1", "close_lag_2", "close_lag_3", "returns_lag_1", "returns_lag_2",                          "close_mean_5", "close_mean_10", "close_std_10", "day_of_week", "month"]        feature_columns = [col for col in feature_columns if col in df.columns]    return df[feature_columns], feature_columnsdef create_sequences(data, target, seq_length=60, forecast_horizon=1):    X, y = [], []    for i in range(len(data) - seq_length - forecast_horizon + 1):        X.append(data[i:i + seq_length])        y.append(target[i + seq_length:i + seq_length + forecast_horizon])    return np.array(X), np.array(y)def scale_data(train_df, test_df, feature_columns, target_column="returns"):    feature_scaler = MinMaxScaler(feature_range=(0, 1))    target_scaler = MinMaxScaler(feature_range=(0, 1))    train_features_scaled = feature_scaler.fit_transform(train_df[feature_columns].values)    train_target_scaled = target_scaler.fit_transform(train_df[[target_column]].values).ravel()    test_features_scaled = feature_scaler.transform(test_df[feature_columns].values)    test_target_scaled = target_scaler.transform(test_df[[target_column]].values).ravel()    return train_features_scaled, train_target_scaled, test_features_scaled, test_target_scaled, feature_scaler, target_scaler